**Lab type:** prompt  
**Course:** ML302 — Transformer Models & Fine-Tuning  
**Lesson:** Fine-tuning Strategies  
**Task:** For each of three realistic scenarios, apply the strategy decision framework from the lesson, then compare a weak AI prompt (which defaults to full fine-tuning) against a strong prompt (which specifies the correct constraints). Finally, audit an AI-generated recommendation against the four-question decision process.

In [ ]:
# No model downloads needed for this lab — all tasks are prompt-writing and analysis.
# The 'scenario context' cells below are runnable and print the key parameters
# for each scenario to keep them visible as you write prompts.
print('Fine-tuning Strategies Lab — no GPU required.')

## How this lab works

Each task presents a realistic fine-tuning scenario with concrete constraints (dataset size, compute, task similarity). You will:

1. **Analyse the scenario** using the four-question decision process from the lesson.
2. **Run the weak-prompt cell** — this shows what a typical AI tool recommends when given a vague request.
3. **Write your own strong prompt** in the designated cell, explicitly encoding the constraints.
4. **Paste and annotate the AI output** in the markdown cell below.

For Tasks 1 and 2, paste the AI output into the markdown cell. For Task 3 you will audit a pre-written AI recommendation.

## Task 1: Medical Named Entity Recognition (NER)

**Scenario parameters:**

In [ ]:
scenario_1 = {
    'task': 'Named entity recognition — identify drug names and dosages in clinical notes',
    'labelled_examples': 180,
    'base_model': 'bert-base-uncased (110M params)',
    'compute': 'Single RTX 3090 (24 GB VRAM)',
    'domain_overlap_with_pretraining': 'Low — clinical notes use specialised terminology not common in web text',
    'must_preserve_general_capability': False,
    'time_budget': '4 hours training max',
}
for k, v in scenario_1.items():
    print(f'{k}: {v}')

**Apply the decision framework** (answer each question before writing your prompt):

1. How much data? → 
2. How different is the task from pretraining? → 
3. What compute is available? → 
4. Must general capability be preserved? → 

**Strategy I would choose:** *(one of: zero-shot / few-shot ICL / feature extraction / PEFT/LoRA / full fine-tuning)*

In [ ]:
# WEAK PROMPT OUTPUT
# Prompt given: 'Help me fine-tune a BERT model for medical NER on my clinical notes dataset.'
#
# A typical AI response defaults to full fine-tuning:

weak_prompt_code = '''
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained('bert-base-uncased', num_labels=5)

training_args = TrainingArguments(
    output_dir='./ner_results',
    num_train_epochs=10,          # full training, many epochs
    per_device_train_batch_size=16,
    learning_rate=5e-5,           # default, no adjustment for tiny dataset
    # No weight_decay, no evaluation, no early stopping
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)
trainer.train()
'''
print('WEAK PROMPT OUTPUT (full fine-tuning on 180 examples):')
print(weak_prompt_code)

**What the weak prompt gets wrong:** The weak prompt uses full fine-tuning on 180 examples — exactly the scenario where catastrophic forgetting and severe overfitting are expected. With 180 examples and a 110M parameter model, the model has over 600K parameters per training example. *(Add your own analysis of the specific risks below.)*

*(Your analysis here.)*

<details>
<summary>🔑 Reveal analysis — Task 1 (weak prompt problems)</summary>

**Overfitting risk:** With 180 labelled examples and 110M parameters, the model has roughly 610,000 parameters per example. Full fine-tuning updates all of them, so the model will almost certainly memorise training examples rather than generalise — evidenced by near-zero training loss after a few epochs while validation metrics plateau or drop.

**No regularisation:** The weak prompt uses `learning_rate=5e-5` (a full fine-tuning default, which is large for a tiny dataset), no weight decay, no early stopping, and no evaluation schedule. These omissions guarantee overfitting without any signal to stop training.

**Correct strategy:** Feature extraction or LoRA (rank 8–16) — freeze the 110M base weights entirely and train only the classification head (or a small set of adapter matrices). This dramatically reduces effective parameter count, constrains the hypothesis space to what the pretrained representations already know, and typically converges in minutes on 180 examples.

</details>

In [ ]:
# Write your strong prompt in the comment below, then paste the AI output
# in the markdown cell that follows.
#
# Your strong prompt should encode:
# - exact dataset size
# - compute constraint
# - domain shift severity
# - required strategy and why
#
# STRONG PROMPT (write it here):
# -----------------------------------------------------------------------
#
# -----------------------------------------------------------------------

print('Write your strong prompt in the comment above, then paste AI output below.')

<details>
<summary>🔑 Model prompt — Task 1</summary>

**Example strong prompt:**

> I need to fine-tune `bert-base-uncased` (110M params) for token classification (NER) to identify drug names and dosages in clinical notes. I have only 180 labelled training examples. The domain is specialised clinical text with low overlap to BERT's web-text pretraining. I have one RTX 3090 (24 GB) and a 4-hour training budget. Full fine-tuning is inappropriate at this dataset size — I need a parameter-efficient approach. Please write a HuggingFace Trainer setup using LoRA (via `peft`) with rank 8, alpha 16, targeting the query and value projection matrices. Include: proper `weight_decay` and a `linear` LR scheduler, per-epoch evaluation, and early stopping with `patience=3`. The output should be a `TokenClassification` head over the frozen base + LoRA adapters.

**Why it's strong:** It encodes every decision-relevant constraint (dataset size, domain shift, compute budget, strategy choice) and specifies the exact implementation path (LoRA rank, target modules, scheduler, early stopping). An AI tool given this prompt cannot default to full fine-tuning because the instruction explicitly rules it out and explains why.

</details>

**AI output from strong prompt:** *(Paste AI-generated code here)*

```python
# paste here
```

**What changed compared to the weak output?** *(Note which constraints the AI respected and which it missed.)*

## Task 2: Code Completion Model (50K Examples)

**Scenario parameters:**

In [ ]:
scenario_2 = {
    'task': 'Code completion for Python — predict next token given prior code context',
    'labelled_examples': 52_000,
    'base_model': 'GPT-2 medium (345M params, decoder-only)',
    'compute': '2x A100 40GB (80 GB total VRAM)',
    'domain_overlap_with_pretraining': 'Moderate — GPT-2 saw some code but mostly prose',
    'must_preserve_general_capability': False,
    'goal': 'Maximise code completion accuracy; general language ability not needed',
}
for k, v in scenario_2.items():
    print(f'{k}: {v}')

**Apply the decision framework**:

1. How much data? → 
2. How different is the task from pretraining? → 
3. What compute is available? → 
4. Must general capability be preserved? → 

**Strategy I would choose:** 

**One sentence justification:** 

In [ ]:
# Write your prompt for this scenario in the comment below.
# The prompt should produce a TrainingArguments / Trainer or raw loop
# that reflects the correct strategy for this scenario.
#
# PROMPT:
# -----------------------------------------------------------------------
#
# -----------------------------------------------------------------------

print('Write your prompt above.')

<details>
<summary>🔑 Model prompt — Task 2</summary>

**Decision framework:**
1. **Data volume:** 52,000 examples — above the ~10,000 threshold where full fine-tuning becomes viable.
2. **Task similarity:** Moderate domain shift (GPT-2 saw some code but mostly prose). Full fine-tuning allows the model to update all weights toward code syntax and idioms.
3. **Compute:** 2× A100 (80 GB total) — sufficient for full fine-tuning of GPT-2 medium (345M params) with gradient accumulation.
4. **Preserve general capability:** No — code completion accuracy is the only goal.

**Strategy:** Full fine-tuning.

**Example strong prompt:**

> Fine-tune `gpt2-medium` (345M parameters, decoder-only) on 52,000 Python code completion examples. General language capability does not need to be preserved — maximising code token prediction accuracy is the only goal. Available compute: 2× A100 40GB. Please write a raw PyTorch training loop (no Trainer abstraction) using: AdamW with `lr=2e-5` and `weight_decay=0.01`, linear warmup over 500 steps then cosine decay, gradient clipping at 1.0, mixed precision (bf16), and per-epoch validation perplexity logging. Unfreeze all weights from epoch 0. Save the best checkpoint by validation perplexity.

**Why it's strong:** States that general capability can be discarded (unlocking full fine-tuning), specifies compute so the model can size batch sizes correctly, and requests standard code-domain hyperparameters (lower LR than default, cosine decay, gradient clipping).

</details>

**AI output:** *(Paste here)*

```python
# paste here
```

**Evaluation:** Did the AI choose the correct strategy given 50K examples and the requirement to maximise task accuracy? What would you change?

## Task 3: General-Purpose Assistant — Preserve Capability

**Scenario:** A company wants to fine-tune LLaMA-3-8B on 3,000 internal customer support conversations to make it follow their response format (structured JSON with `answer`, `confidence`, `escalate` fields). After fine-tuning, the assistant must still handle novel queries that were not in the training set, and must not degrade on general reasoning tasks.

An AI tool was given this brief and produced the recommendation below. Audit it against the four-question decision framework.

In [ ]:
# Pre-written AI recommendation to audit
ai_recommendation = '''
I recommend full fine-tuning of LLaMA-3-8B on your 3,000 customer support examples.

Here is the setup:
- Unfreeze all model layers
- Train for 5 epochs with lr=5e-5
- batch_size=4 with gradient accumulation=8 (effective batch=32)
- No evaluation during training (add it later if needed)
- Save the final checkpoint

This gives the model maximum flexibility to learn your JSON output format.
After fine-tuning, test the model on a few manual examples to verify the format.
'''
print(ai_recommendation)

**Audit checklist** — work through each question and flag every problem in the AI recommendation:

1. **Data volume check** (3,000 examples): Is full fine-tuning appropriate at this dataset size? What rule of thumb from the lesson applies?
   
   *(Your answer.)*

2. **Task similarity** (response format adaptation): Does adapting output format require updating all 8B weights, or is this a case where LoRA rank 8–16 would suffice?
   
   *(Your answer.)*

3. **General capability preservation**: The company explicitly requires no degradation on novel queries. What does full fine-tuning at 5 epochs on 3,000 examples do to the model's general capability? What would you use instead?
   
   *(Your answer.)*

4. **Evaluation strategy**: 'No evaluation during training' appears in the recommendation. What specific risk does this create for detecting catastrophic forgetting, and what should be added?
   
   *(Your answer.)*

**Revised recommendation in one paragraph:**

*(Write your corrected strategy here.)*

<details>
<summary>🔑 Reveal audit answers — Task 3</summary>

**1. Data volume check:** No — full fine-tuning is not appropriate at 3,000 examples for an 8B-parameter model. The rule of thumb from the lesson: below ~10,000 examples, use PEFT/LoRA. With 8B parameters and 3,000 examples, the model has ~2.7M parameters per training example, virtually guaranteeing memorisation.

**2. Task similarity:** Adapting output format (structured JSON) is a lightweight surface-level change — the model only needs to learn a consistent output template, not new world knowledge. LoRA at rank 8–16 targeting the attention projection matrices is more than sufficient; there is no need to update all 8B weights for this.

**3. General capability preservation:** Full fine-tuning at 5 epochs on 3,000 examples will cause catastrophic forgetting — the model overwrites weights that encode general reasoning in order to fit the small training distribution. The company's requirement for "no degradation on novel queries" makes this fatal. The correct approach is LoRA: adapters are added to a frozen base, so the base weights (and their general capabilities) are untouched.

**4. Evaluation strategy:** Skipping evaluation means there is no signal for when catastrophic forgetting begins. At minimum: a fixed held-out set of general-reasoning benchmarks (e.g., 200 MMLU questions) should be evaluated each epoch alongside the target-format accuracy metric. A significant drop in general-reasoning score is the early-stopping signal for forgetting.

**Revised recommendation:** Use LoRA (rank 16, alpha 32) on LLaMA-3-8B, targeting the Q/V attention projections. Train for 3–5 epochs on the 3,000 examples with `lr=2e-4`, weight decay 0.01, and a 10% warmup. Evaluate format accuracy and a MMLU probe set each epoch; stop training if general-reasoning accuracy drops more than 2 points. The frozen base preserves all general capability; the adapters encode only the JSON output format.

</details>

## Summary

> **Complete each sentence.**

1. When dataset size is below 10,000 examples, the default strategy should be _____ rather than full fine-tuning because _____.
2. Full fine-tuning is appropriate when _____ and _____ are both true.
3. Benchmark contamination matters because _____ — the minimum check is _____.

<details>
<summary>🔑 Reveal summary answers</summary>

1. When dataset size is below 10,000 examples, the default strategy should be **PEFT/LoRA (or feature extraction)** rather than full fine-tuning because **updating all model weights on a small dataset causes catastrophic overfitting — the model memorises the training set rather than generalising, and any general capability encoded in the base weights is overwritten**.

2. Full fine-tuning is appropriate when **the dataset is large enough (typically 10,000+ examples)** and **preserving the model's general pretraining capability is not required** (i.e., the task fully replaces the original use case).

3. Benchmark contamination matters because **a fine-tuned model may have seen benchmark questions in its fine-tuning data, producing inflated scores that don't reflect real-world performance** — the minimum check is **running the benchmark on the base model before fine-tuning and comparing to the fine-tuned result to confirm any gain is genuine and not already present in the base**.

</details>